# 5. Incompatibility number $\kappa$ and further checks of Sections 5–7

This notebook reproduces

* the **incompatibility number** $\kappa$ computed by Algorithm 1 of the paper (thresholds $\eta^\star(C)$ by
  semidefinite programming, then the covering linear program of Definition 3), with $\kappa_{\mathrm{part}}$, the cover
  number $G$, $\Lambda$ and the subfamily bound $\max_{\mathcal A'}|\mathcal A'|/\Lambda(\mathcal A')$ of Proposition 5;
* the **5-cycle example** of Section 5 ($\kappa = K/\Lambda = 5/2 < \kappa_{\mathrm{part}} = 2.639 < G = 3$);
* the **family test of Open problem 2** (Section 7): 30 random families of five or six two-qubit Pauli strings;
* the **Born-rule check** of the binomial model and the **typical-instance** experiment (Section 6, Q2 and Q3);
* the **repeatability of $B_{0.9}$** over five independent seeds (Section 6, Q2).

Each part compares its output with the paper data in `reference/`. The $\kappa$ computations are deterministic up to
solver tolerance and are compared exactly. The Monte Carlo parts are bit-identical to the paper data only with the
NumPy version used for the paper (random-variate algorithms differ between versions); otherwise they are compared
in units of their Monte Carlo standard error, and the numbers quoted in the paper are recomputed.

**Run time:** about ten minutes. Set `RECOMPUTE = False` to reuse the results stored in `results/`.

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RESULTS, REFERENCE = ROOT / "results", ROOT / "reference"
RESULTS.mkdir(exist_ok=True)

from copycost import kappa as kp, checks, selection as sel

RECOMPUTE = True


KAPPA_FIELDS = ["K", "G", "kappa", "kappa_part", "Lambda", "K_over_Lambda", "eta_full", "kappa_sharp_only",
                "subfamily_bound"]


def compare_kappa(new, old, tol=1e-4):
    # every quantity of Section 5; the optimal cover itself is not unique (degenerate LP), so it is not compared
    bad = [(f, new[f], old[f]) for f in KAPPA_FIELDS if not np.isclose(new[f], old[f], atol=tol)]
    return bad


def mc_diff(new, old, se):
    # largest |new - old| in units of the standard error of a difference of two independent runs
    d = np.abs(np.asarray(new, float) - np.asarray(old, float))
    return float(d.max()), float((d / (np.sqrt(2) * np.asarray(se, float))).max())

## Algorithm 1 on small examples

A qubit pair at angle $\theta$ (Proposition 4, $\kappa = 1+\sin\theta$) and the block $X, Y, Z$ (Proposition 3,
$\kappa = G = 3$). `kp.thresholds(A)` is stage 1 (one SDP per non-commuting group), `kp.kappa_lp` stage 2.

In [2]:
rows = []
X, Z = kp.pauli("X"), kp.pauli("Z")
for th in [0, 20, 40, 60, 90]:
    t = np.radians(th)
    A = [Z, np.cos(t) * Z + np.sin(t) * X]
    rows.append({"family": f"pair, theta={th} deg", "kappa (Alg. 1)": kp.kappa_lp(2, kp.thresholds(A))[0],
                 "closed form": 1 + np.sin(t)})
A = [kp.pauli(s) for s in "XYZ"]
rows.append({"family": "block X, Y, Z", "kappa (Alg. 1)": kp.kappa_lp(3, kp.thresholds(A))[0], "closed form": 3.0})
pd.DataFrame(rows).style.format(precision=5)

,family,kappa (Alg. 1),closed form
0,"pair, theta=0 deg",1.00000,1.00000
1,"pair, theta=20 deg",1.34202,1.34202
2,"pair, theta=40 deg",1.64279,1.64279
3,"pair, theta=60 deg",1.86603,1.86603
4,"pair, theta=90 deg",2.00000,2.00000
5,"block X, Y, Z",3.00000,3.00000


## 5-cycle (Section 5) and random families (Open problem 2)

One generator seeded 11 is shared by the two parts and used in the order of the original script, which reproduces
the paper data exactly. The random-family test solves one SDP for every non-commuting subset of each family and
computes $\Lambda$ for every subfamily, which takes a few minutes.

In [3]:
OUT = RESULTS / "kappa_cover.json"
if RECOMPUTE or not OUT.exists():
    t0 = time.time()
    rng = np.random.default_rng(11)
    res = {"five_cycle": kp.analyse(kp.FIVE_CYCLE, rng)}
    res["random_families"] = kp.random_families(rng)
    OUT.write_text(json.dumps(res, indent=1))
    print(f"computed in {time.time() - t0:.0f} s")
res = json.loads(OUT.read_text())
ref = json.loads((REFERENCE / "kappa_cover.json").read_text())
bad = compare_kappa(res["five_cycle"], ref["five_cycle"])
print("5-cycle:", "all quantities identical to the paper data" if not bad else bad)
bad = [(i, b) for i, (n, o) in enumerate(zip(res["random_families"], ref["random_families"])) if (b := compare_kappa(n, o))]
same_family = all(n["family"] == o["family"] for n, o in zip(res["random_families"], ref["random_families"]))
print(f"random families: same {len(ref['random_families'])} families drawn: {same_family}; "
      + ("all quantities identical to the paper data" if not bad else f"differences: {bad}"))

computed in 267 s
5-cycle: all quantities identical to the paper data
random families: same 30 families drawn: True; all quantities identical to the paper data


In [4]:
c = res["five_cycle"]
print(f"5-cycle {' '.join(c['family'])}:  G = {c['G']},  kappa = {c['kappa']:.4f},  kappa_part = {c['kappa_part']:.4f},  "
      f"eta*(all five) = {c['eta_full']:.4f},  Lambda = {c['Lambda']:.4f},  K/Lambda = {c['K_over_Lambda']:.4f}")
print("optimal cover:", ", ".join(f"{'+'.join(g['group'])} (w={g['weight']:.2f})" for g in c["cover"]))
fams, tol = res["random_families"], 1e-3
print(f"\nrandom families: {len(fams)}")
print(f"  kappa == max_A' |A'|/Lambda(A')  (conjecture of Open problem 2): {sum(abs(f['kappa'] - f['subfamily_bound']) < tol for f in fams)}")
print(f"  kappa == G                                                     : {sum(abs(f['kappa'] - f['G']) < tol for f in fams)}")
print(f"  K/Lambda < kappa (subfamily form needed)                       : {sum(f['kappa'] - f['K_over_Lambda'] > tol for f in fams)}")
print(f"  largest kappa / (K/Lambda)                                     : {max(f['kappa'] / f['K_over_Lambda'] for f in fams):.2f}")

5-cycle IX XI IY ZY YX:  G = 3,  kappa = 2.5000,  kappa_part = 2.6393,  eta*(all five) = 0.6155,  Lambda = 2.0000,  K/Lambda = 2.5000
optimal cover: IX+XI (w=0.50), IX+YX (w=0.50), XI+IY (w=0.50), IY+ZY (w=0.50), ZY+YX (w=0.50)

random families: 30
  kappa == max_A' |A'|/Lambda(A')  (conjecture of Open problem 2): 30
  kappa == G                                                     : 30
  K/Lambda < kappa (subfamily form needed)                       : 28
  largest kappa / (K/Lambda)                                     : 2.00


## Born-rule check and typical instances (Section 6)

The selection simulator draws each candidate's statistic from a binomial law. Here the outcome probabilities are
computed from the measurement operators by the Born rule, and the three outcomes of the joint POVM on a qubit are
drawn together. One generator seeded 20261117 is shared by the two parts.

In [5]:
OUT = RESULTS / "born_rule_check.json"
if RECOMPUTE or not OUT.exists():
    t0 = time.time()
    rng = np.random.default_rng(20261117)
    born = {"hard": checks.hard_instance_check(rng)}
    born["typical"] = checks.typical_instances(rng)
    OUT.write_text(json.dumps(born, indent=1))
    print(f"computed in {time.time() - t0:.0f} s")
born = json.loads(OUT.read_text())
ref = json.loads((REFERENCE / "born_rule_check.json").read_text())
T = born["hard"]["T"]
for L in ["grouped", "joint"]:
    for kind in ["born", "model"]:
        d, z = mc_diff(born["hard"][kind][L], ref["hard"][kind][L], 0.5 / np.sqrt(T))
        print(f"hard instances, {L:7s} {kind:5s}: max |this run - paper data| = {d:.4f} (<= {z:.1f} s.e.)")
for L in ["grouped", "joint"]:
    se = born["typical"][L + "_se"]
    d, z = mc_diff(born["typical"][L], ref["typical"][L], se)
    print(f"typical instances, {L:7s}: max |this run - paper data| = {d:.5f} (<= {z:.1f} s.e.)")

computed in 3 s
hard instances, grouped born : max |this run - paper data| = 0.0000 (<= 0.0 s.e.)
hard instances, grouped model: max |this run - paper data| = 0.0000 (<= 0.0 s.e.)
hard instances, joint   born : max |this run - paper data| = 0.0112 (<= 1.2 s.e.)
hard instances, joint   model: max |this run - paper data| = 0.0105 (<= 1.2 s.e.)
typical instances, grouped: max |this run - paper data| = 0.00026 (<= 0.9 s.e.)
typical instances, joint  : max |this run - paper data| = 0.00061 (<= 0.7 s.e.)


In [6]:
h = born["hard"]
print(f"hard instances (K = {h['K']}): max |Born rule - binomial model| = "
      f"{max(h['max_abs_diff_grouped'], h['max_abs_diff_joint']):.3f}, two standard errors = {h['two_se_bound']:.3f}")
t = born["typical"]
G_, m_ = 3, t["m"]
pd.DataFrame({"B": t["B"], "grouped": t["grouped"], "joint": t["joint"],
              "joint / grouped - 1 (%)": 100 * (np.array(t["joint"]) / np.array(t["grouped"]) - 1),
              "same, paper data (%)": 100 * (np.array(ref["typical"]["joint"]) / np.array(ref["typical"]["grouped"]) - 1),
              "Theorem 1 bound": [3 * np.sqrt(G_ * np.log(2 * 3 * m_) / B) for B in t["B"]]}).style.format(precision=4)

hard instances (K = 30): max |Born rule - binomial model| = 0.014, two standard errors = 0.018


,B,grouped,joint,joint / grouped - 1 (%),"same, paper data (%)",Theorem 1 bound
0,90,0.0446,0.0457,2.5789,3.7398,1.1083
1,300,0.0185,0.0203,9.5223,8.7481,0.6070
2,900,0.0070,0.0081,15.5271,13.0089,0.3505
3,3000,0.0023,0.0027,17.0551,17.0535,0.1920
4,9000,0.0008,0.0009,19.0848,11.4856,0.1108


## Repeatability of $B_{0.9}$ (Section 6)

The bisection of Algorithm 6, rerun for both non-adaptive learners with independent seeds $1000 + 17r + G$.

In [7]:
OUT = RESULTS / "b90_reps.json"
if RECOMPUTE or not OUT.exists():
    t0 = time.time()
    reps = {"eps": sel.EPS_DEFAULT, "K": sel.K_DEFAULT, "reps": 5, "T": 6000, "G": [1, 2, 3, 5, 6, 9, 10, 15, 18]}
    for L in ["grouped", "joint"]:
        reps[L] = sel.b90_repeatability(L)
    OUT.write_text(json.dumps(reps, indent=1))
    print(f"computed in {time.time() - t0:.0f} s")
reps = json.loads(OUT.read_text())
ref = json.loads((REFERENCE / "b90_reps.json").read_text())
for L in ["grouped", "joint"]:
    a, b = np.array(reps[L]["b90"]), np.array(ref[L]["b90"])
    print(f"{L:7s}: identical to the paper data: {np.array_equal(a, b)};  max relative difference of B_0.9 "
          f"{100 * np.max(np.abs(a - b) / b):.1f}% (bisection resolution 2%)")
print("paper (Section 6): slope 1500 +- 7 (grouped), 1498 +- 14 (joint); relative s.d. <= 1.8%; normalized cost 3.23-3.43")
for L in ["grouped", "joint"]:
    r = reps[L]
    print(f"{L:7s}: slope {r['slope_mean']:.0f} +- {r['slope_sd']:.0f} copies per unit G;  relative s.d. of each "
          f"B_0.9 at most {100 * r['max_rel_sd']:.1f}%;  normalized cost {r['norm_min']:.2f} to {r['norm_max']:.2f}")

computed in 22 s
grouped: identical to the paper data: False;  max relative difference of B_0.9 2.4% (bisection resolution 2%)
joint  : identical to the paper data: True;  max relative difference of B_0.9 0.0% (bisection resolution 2%)
paper (Section 6): slope 1500 +- 7 (grouped), 1498 +- 14 (joint); relative s.d. <= 1.8%; normalized cost 3.23-3.43
grouped: slope 1502 +- 5 copies per unit G;  relative s.d. of each B_0.9 at most 1.5%;  normalized cost 3.27 to 3.43
joint  : slope 1498 +- 14 copies per unit G;  relative s.d. of each B_0.9 at most 1.8%;  normalized cost 3.23 to 3.42
